In [1]:
import pandas as pd
import numpy as np

In [2]:
DATA_PATH = "../data/processed/null_handled_columns_dropped.csv"

df = pd.read_csv(
    DATA_PATH,
    low_memory=False
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!
Shape: (156985, 9)

Columns:
['Restaurant_ID', 'Facility Type', 'Risk', 'Address', 'Inspection Date', 'Inspection Type', 'Results', 'Violations', 'Location']


In [3]:
print("Location dtype:", df["Location"].dtype)

print("\nMissing Location values:")
print(df["Location"].isna().sum())

print("\nSample Location values:")
display(df["Location"].head(20))

Location dtype: str

Missing Location values:
0

Sample Location values:


0      (41.88075715864721, -87.6347092983425)
1      (41.88075715864721, -87.6347092983425)
2      (41.88075715864721, -87.6347092983425)
3      (41.88075715864721, -87.6347092983425)
4      (41.88075715864721, -87.6347092983425)
5      (41.88075715864721, -87.6347092983425)
6      (41.88075715864721, -87.6347092983425)
7      (41.88075715864721, -87.6347092983425)
8      (41.88075715864721, -87.6347092983425)
9      (41.88075715864721, -87.6347092983425)
10     (41.88075715864721, -87.6347092983425)
11     (41.88075715864721, -87.6347092983425)
12     (41.88075715864721, -87.6347092983425)
13     (41.88075715864721, -87.6347092983425)
14     (41.88075715864721, -87.6347092983425)
15    (41.88039583825962, -87.62450172159464)
16    (41.88039583825962, -87.62450172159464)
17    (41.88039583825962, -87.62450172159464)
18    (41.88039583825962, -87.62450172159464)
19    (41.88039583825962, -87.62450172159464)
Name: Location, dtype: str

In [4]:
id="e4x7kp"
df[["Latitude", "Longitude"]] = (
    df["Location"]
    .str.strip("()")
    .str.split(",", expand=True)
    .astype(float)
)

print("Latitude/Longitude extracted successfully!")

print("\nMissing Latitude:", df["Latitude"].isna().sum())
print("Missing Longitude:", df["Longitude"].isna().sum())

display(df[["Location", "Latitude", "Longitude"]].head(10))

Latitude/Longitude extracted successfully!

Missing Latitude: 0
Missing Longitude: 0


,Location,Latitude,Longitude
0,"(41.88075715864721, -87.6347092983425)",41.880757,-87.634709
1,"(41.88075715864721, -87.6347092983425)",41.880757,-87.634709
2,"(41.88075715864721, -87.6347092983425)",41.880757,-87.634709
3,"(41.88075715864721, -87.6347092983425)",41.880757,-87.634709
4,"(41.88075715864721, -87.6347092983425)",41.880757,-87.634709
5,"(41.88075715864721, -87.6347092983425)",41.880757,-87.634709
6,"(41.88075715864721, -87.6347092983425)",41.880757,-87.634709
7,"(41.88075715864721, -87.6347092983425)",41.880757,-87.634709
8,"(41.88075715864721, -87.6347092983425)",41.880757,-87.634709
9,"(41.88075715864721, -87.6347092983425)",41.880757,-87.634709


In [5]:
id_location_counts = (
    df.groupby("Restaurant_ID")
      .apply(
          lambda x: x[["Latitude", "Longitude"]]
          .drop_duplicates()
          .shape[0]
      )
)

print("Unique Restaurant_IDs:", len(id_location_counts))

print("\nRestaurant_IDs with exactly 1 coordinate pair:",
      (id_location_counts == 1).sum())

print("Restaurant_IDs with multiple coordinate pairs:",
      (id_location_counts > 1).sum())

print("Maximum coordinate pairs for one Restaurant_ID:",
      id_location_counts.max())

Unique Restaurant_IDs: 22291

Restaurant_IDs with exactly 1 coordinate pair: 22246
Restaurant_IDs with multiple coordinate pairs: 45
Maximum coordinate pairs for one Restaurant_ID: 2


In [6]:
multi_location_ids = id_location_counts[
    id_location_counts > 1
].index

multi_location_details = (
    df[df["Restaurant_ID"].isin(multi_location_ids)]
    [["Restaurant_ID", "Inspection Date", "Address",
      "Latitude", "Longitude"]]
    .drop_duplicates()
    .sort_values(
        ["Restaurant_ID", "Inspection Date"]
    )
)

print("Restaurant_IDs with multiple coordinate pairs:",
      len(multi_location_ids))

display(multi_location_details)

Restaurant_IDs with multiple coordinate pairs: 45


,Restaurant_ID,Inspection Date,Address,Latitude,Longitude
9295,23120.0,2011-07-27,251 E HURON ST,41.894903,-87.620744
9297,23120.0,2013-11-20,257 E HURON ST,41.894906,-87.620520
9298,23120.0,2015-11-18,257 E HURON ST,41.894906,-87.620520
9296,23120.0,2016-11-14,257 E HURON ST,41.894906,-87.620520
9299,23120.0,2018-10-19,257 E HURON ST,41.894906,-87.620520
...,...,...,...,...,...
154087,2983053.0,2026-07-28,11601 W TOUHY AVE,42.008751,-87.906874
154200,2983595.0,2024-08-07,10000 W OHARE ST,41.976201,-87.905309
154202,2983595.0,2024-08-07,11601 W TOUHY AVE,42.008751,-87.906874
154201,2983595.0,2026-01-09,10000 W OHARE ST,41.976201,-87.905309


In [7]:
from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    return 2 * R * atan2(sqrt(a), sqrt(1 - a))


multi_location_distance = []

for restaurant_id in multi_location_ids:

    locations = (
        df[df["Restaurant_ID"] == restaurant_id]
        [["Latitude", "Longitude"]]
        .drop_duplicates()
        .values
    )

    lat1, lon1 = locations[0]
    lat2, lon2 = locations[1]

    distance = haversine_distance(
        lat1, lon1,
        lat2, lon2
    )

    multi_location_distance.append({
        "Restaurant_ID": restaurant_id,
        "Latitude_1": lat1,
        "Longitude_1": lon1,
        "Latitude_2": lat2,
        "Longitude_2": lon2,
        "Distance_meters": distance
    })

multi_location_distance_df = (
    pd.DataFrame(multi_location_distance)
    .sort_values("Distance_meters")
)

display(multi_location_distance_df)

,Restaurant_ID,Latitude_1,Longitude_1,Latitude_2,Longitude_2,Distance_meters
16,2125423.0,41.736226,-87.629260,41.736218,-87.629310,4.256964
21,2290163.0,41.877259,-87.635041,41.877213,-87.635041,5.115310
36,2589451.0,41.755233,-87.741406,41.755182,-87.741404,5.742631
35,2551263.0,41.816597,-87.700393,41.816563,-87.700448,5.926303
20,2232394.0,41.935196,-87.648951,41.935251,-87.648953,6.100255
18,2157629.0,41.924153,-87.744392,41.924152,-87.744470,6.439491
40,2776230.0,41.850290,-87.631812,41.850232,-87.631811,6.447554
15,2124632.0,41.968140,-87.727263,41.968139,-87.727341,6.470710
14,2003847.0,42.008037,-87.664896,42.008036,-87.664974,6.479301
37,2609815.0,41.947118,-87.655381,41.947117,-87.655460,6.511000


In [8]:
distance_bins = pd.cut(
    multi_location_distance_df["Distance_meters"],
    bins=[0, 10, 25, 50, 100, 250, 1000, np.inf],
    labels=[
        "<10 m",
        "10–25 m",
        "25–50 m",
        "50–100 m",
        "100–250 m",
        "250 m–1 km",
        ">1 km"
    ],
    include_lowest=True
)

distance_distribution = (
    distance_bins
    .value_counts()
    .sort_index()
    .rename("Restaurant_ID_count")
    .reset_index()
)

display(distance_distribution)

print(
    "\nIDs with coordinate difference <= 25 m:",
    (multi_location_distance_df["Distance_meters"] <= 25).sum()
)

print(
    "IDs with coordinate difference > 1 km:",
    (multi_location_distance_df["Distance_meters"] > 1000).sum()
)

,Distance_meters,Restaurant_ID_count
0,<10 m,17
1,10–25 m,6
2,25–50 m,8
3,50–100 m,4
4,100–250 m,6
5,250 m–1 km,0
6,>1 km,4



IDs with coordinate difference <= 25 m: 23
IDs with coordinate difference > 1 km: 4


In [9]:
location_pair_counts = (
    df.groupby(
        ["Restaurant_ID", "Latitude", "Longitude"]
    )
    .size()
    .reset_index(name="inspection_count")
)

print("Unique Restaurant_ID + coordinate pairs:",
      len(location_pair_counts))

print("\nTotal inspection rows:",
      len(df))

print("\nCoordinate pairs with 1 inspection:",
      (location_pair_counts["inspection_count"] == 1).sum())

print("Coordinate pairs with 2+ inspections:",
      (location_pair_counts["inspection_count"] >= 2).sum())

print("\nMaximum inspections for one Restaurant_ID + coordinate pair:",
      location_pair_counts["inspection_count"].max())

Unique Restaurant_ID + coordinate pairs: 22336

Total inspection rows: 156985

Coordinate pairs with 1 inspection: 4152
Coordinate pairs with 2+ inspections: 18184

Maximum inspections for one Restaurant_ID + coordinate pair: 61


In [10]:
pairs_per_restaurant = (
    location_pair_counts
    .groupby("Restaurant_ID")
    .size()
    .rename("coordinate_pair_count")
)

print("Restaurant_IDs:", len(pairs_per_restaurant))

print("\nRestaurant_IDs with 1 coordinate pair:",
      (pairs_per_restaurant == 1).sum())

print("Restaurant_IDs with 2 coordinate pairs:",
      (pairs_per_restaurant == 2).sum())

print("Restaurant_IDs with more than 2 coordinate pairs:",
      (pairs_per_restaurant > 2).sum())

print("\nMaximum coordinate pairs for one Restaurant_ID:",
      pairs_per_restaurant.max())

Restaurant_IDs: 22291

Restaurant_IDs with 1 coordinate pair: 22246
Restaurant_IDs with 2 coordinate pairs: 45
Restaurant_IDs with more than 2 coordinate pairs: 0

Maximum coordinate pairs for one Restaurant_ID: 2


In [11]:
df["Latitude_4dp"] = df["Latitude"].round(4)
df["Longitude_4dp"] = df["Longitude"].round(4)

rounded_location_counts = (
    df.groupby(
        ["Restaurant_ID", "Latitude_4dp", "Longitude_4dp"]
    )
    .size()
    .reset_index(name="inspection_count")
)

print("Unique exact coordinate pairs:",
      len(location_pair_counts))

print("Unique coordinate pairs after rounding to 4 decimals:",
      len(rounded_location_counts))

print(
    "\nReduction in location pairs:",
    len(location_pair_counts) - len(rounded_location_counts)
)

Unique exact coordinate pairs: 22336
Unique coordinate pairs after rounding to 4 decimals: 22331

Reduction in location pairs: 5


In [12]:
id_location_address_counts = (
    df.groupby(
        ["Restaurant_ID", "Latitude", "Longitude"]
    )["Address"]
    .nunique()
)

print("Restaurant-location pairs:",
      len(id_location_address_counts))

print("\nPairs with exactly 1 address:",
      (id_location_address_counts == 1).sum())

print("Pairs with multiple addresses:",
      (id_location_address_counts > 1).sum())

print("\nMaximum addresses for one restaurant-location pair:",
      id_location_address_counts.max())

Restaurant-location pairs: 22336

Pairs with exactly 1 address: 11883
Pairs with multiple addresses: 10453

Maximum addresses for one restaurant-location pair: 4


In [13]:
df["Restaurant_Location_ID"] = (
    df["Restaurant_ID"].astype(str)
    + "_"
    + df["Latitude"].astype(str)
    + "_"
    + df["Longitude"].astype(str)
)

print("Restaurant-location identifiers created.")

print("\nUnique Restaurant_Location_IDs:",
      df["Restaurant_Location_ID"].nunique())

print("\nTotal rows:",
      len(df))

display(
    df[
        [
            "Restaurant_ID",
            "Latitude",
            "Longitude",
            "Restaurant_Location_ID"
        ]
    ].head(10)
)

Restaurant-location identifiers created.

Unique Restaurant_Location_IDs: 22336

Total rows: 156985


,Restaurant_ID,Latitude,Longitude,Restaurant_Location_ID
0,2.0,41.880757,-87.634709,2.0_41.88075715864721_-87.6347092983425
1,2.0,41.880757,-87.634709,2.0_41.88075715864721_-87.6347092983425
2,2.0,41.880757,-87.634709,2.0_41.88075715864721_-87.6347092983425
3,2.0,41.880757,-87.634709,2.0_41.88075715864721_-87.6347092983425
4,2.0,41.880757,-87.634709,2.0_41.88075715864721_-87.6347092983425
5,2.0,41.880757,-87.634709,2.0_41.88075715864721_-87.6347092983425
6,2.0,41.880757,-87.634709,2.0_41.88075715864721_-87.6347092983425
7,2.0,41.880757,-87.634709,2.0_41.88075715864721_-87.6347092983425
8,2.0,41.880757,-87.634709,2.0_41.88075715864721_-87.6347092983425
9,2.0,41.880757,-87.634709,2.0_41.88075715864721_-87.6347092983425


In [14]:
df = df.rename(columns={
    "Restaurant_ID": "License",
    "Restaurant_Location_ID": "Restaurant_ID"
})

print(df.columns.tolist())

['License', 'Facility Type', 'Risk', 'Address', 'Inspection Date', 'Inspection Type', 'Results', 'Violations', 'Location', 'Latitude', 'Longitude', 'Latitude_4dp', 'Longitude_4dp', 'Restaurant_ID']


In [15]:
df["Restaurant_ID"] = (
    df
    .groupby(["License", "Latitude", "Longitude"], sort=True)
    .ngroup() + 1
)

In [16]:

# Select only the required columns
restaurant_mapping = df[["Restaurant_ID", "DBA Name"]].copy()

# Rename the restaurant name column
restaurant_mapping.rename(
    columns={"DBA Name": "Restaurant_Name"},
    inplace=True
)

# Remove duplicate restaurant IDs, keeping the first occurrence
restaurant_mapping = restaurant_mapping.drop_duplicates(
    subset=["Restaurant_ID"],
    keep="first"
)

# Sort by Restaurant_ID in ascending order
restaurant_mapping = restaurant_mapping.sort_values(
    by="Restaurant_ID",
    ascending=True
)

# Save only these two columns to CSV
restaurant_mapping.to_csv(
    "../data/processed/restaurant_id_name_mapping.csv",
    index=False
)

print("CSV created successfully!")
display(restaurant_mapping.head(10))


KeyError: "['DBA Name'] not in index"